# WorkflowDirector — Acceptance Lab (Comfy 0.39 / Colab T4)

**Pruebas controladas, sin reinstalar Comfy ni descargar modelos.**

Este notebook actualiza **sólo el código del custom node WorkflowDirector** en una revisión experimental, conserva la instalación de Comfy, el directorio de modelos y los flags de inicio, y luego ejecuta pruebas pequeñas con la API de Comfy.

**No ejecutes las celdas de instalación limpia del notebook antiguo:** algunas borran /content/ComfyUI. Detén Auto Queue y termina cualquier ejecución previa antes de empezar.

Orden: 1. Revisar entorno → 2. Cambiar a revisión experimental → 3. Reiniciar únicamente Comfy → 4. Preflight → 5. STRING → 6. IMAGE → 7. LATENT → 8. Falla controlada → 9. Pestañas del navegador. No correr los modelos pesados hasta validar estos pasos.

**No se usa `/free`, `unload_all_models`, `git reset --hard`, Cloudflare ni comandos que eliminen archivos.**

In [ ]:
# 1 — Confirmar Comfy, ruta de WorkflowDirector y cola vacía
import sys, json, subprocess, time, os
from pathlib import Path
from urllib.request import urlopen
import psutil

BASE = "http://127.0.0.1:8188"
WD_DIR = Path("/content/ComfyUI/custom_nodes/ComfyUI-WorkflowDirector")
COMFY_DIR = Path("/content/ComfyUI")
assert (COMFY_DIR / "main.py").is_file(), "No existe ComfyUI en /content/ComfyUI"
assert (WD_DIR / ".git").is_dir(), "No existe el repositorio de WorkflowDirector"
with urlopen(BASE + "/api/jobs?status=pending,in_progress",timeout=10) as r:
    live_jobs = json.load(r).get("jobs", [])
with urlopen(BASE + "/workflowdirector/health",timeout=10) as r:
    wd_health = json.load(r)
assert not live_jobs, "Hay jobs Comfy en curso: no cambiar de código durante una ejecución."
assert not wd_health.get("director_service",{}).get("active_run_id"), "Director sigue activo."

def run_git(*args):
    return subprocess.check_output(["git","-C",str(WD_DIR),*args],text=True).strip()
dirty = run_git("status","--porcelain")
assert not dirty, "WorkflowDirector contiene modificaciones locales. No se hará checkout: guarda antes tus cambios."
original_commit = run_git("rev-parse","HEAD")
Path("/content/workflowdirector_before_acceptance_commit.txt").write_text(original_commit)
print("Comfy actual:",wd_health.get("comfyui_version"))
print("WorkflowDirector original:",original_commit)
print("Cola vacía. No se modificó nada todavía.")

In [ ]:
# 2 — Obtener la rama de aceptación y conservar el commit anterior
TARGET_BRANCH = "feature/colab-acceptance"
subprocess.run(["git","-C",str(WD_DIR),"fetch","origin",TARGET_BRANCH],check=True)
target_commit = run_git("rev-parse","FETCH_HEAD")
assert target_commit != "", "No se obtuvo un commit de pruebas"
assert not run_git("status","--porcelain"), "Hay modificaciones locales inesperadas"
subprocess.run(["git","-C",str(WD_DIR),"checkout","--detach",target_commit],check=True)
assert (WD_DIR/"scripts"/"acceptance_probe.py").is_file()
assert (WD_DIR/"nodes"/"context_nodes.py").is_file()
print("Versión preparada:",run_git("rev-parse","--short","HEAD"))
print("Versión para regresar:",original_commit[:12])
print("No se reinició Comfy todavía.")

In [ ]:
# 3 — Reiniciar SÓLO Comfy preservando su comando, cwd y flags
# Requiere tener una Comfy actualmente ejecutándose en el puerto 8188.
# No reinstala dependencias, no cambia modelos ni flags.
import signal

candidates=[]
for p in psutil.process_iter(["pid","cmdline","cwd"]):
    try:
        cmd=p.info["cmdline"] or []
        if any(x.endswith("/ComfyUI/main.py") or x=="main.py" for x in cmd) and "--port" in cmd and "8188" in cmd:
            candidates.append(p)
    except (psutil.NoSuchProcess,psutil.AccessDenied):
        continue
assert len(candidates)==1, f"Se esperaba un solo Comfy en 8188: {[p.pid for p in candidates]}"
old=candidates[0]
args=old.cmdline()
cwd=old.cwd()
assert "--cache-none" in args, "Tu proceso no tiene --cache-none. Usa tu celda de arranque conocida, con ese flag."
assert any(x.endswith("/ComfyUI/main.py") or x=="main.py" for x in args), "Comando de Comfy desconocido"
with urlopen(BASE + "/api/jobs?status=pending,in_progress",timeout=10) as r:
    assert not json.load(r).get("jobs"), "Comfy tiene trabajos activos"
with urlopen(BASE + "/workflowdirector/health",timeout=10) as r:
    assert not json.load(r).get("director_service",{}).get("active_run_id"), "Director sigue activo"
print("Restauraré exactamente estos argumentos:",args[1:])
print("PWD:",cwd)
old.terminate()
try:
    old.wait(timeout=25)
except psutil.TimeoutExpired:
    raise RuntimeError("Comfy no terminó por SIGTERM. Deténlo manualmente; NO inicié otro proceso.")
logfile=Path("/content/comfyui_workflowdirector_acceptance.log")
loghandle=open(logfile,"w",buffering=1)
env=os.environ.copy()
env.setdefault("WORKFLOWDIRECTOR_OBSERVATION_SECONDS","1.0")
p=subprocess.Popen(args,cwd=cwd,env=env,stdout=loghandle,stderr=subprocess.STDOUT)
loghandle.close()
Path("/content/comfyui_workflowdirector.pid").write_text(str(p.pid))
for i in range(240):
    if p.poll() is not None:
        raise RuntimeError("Comfy se cerró al arrancar:\n"+logfile.read_text(errors="replace")[-6000:])
    try:
        with urlopen(BASE + "/workflowdirector/health",timeout=3) as r:
            health=json.load(r)
        if health.get("director_service",{}).get("ready"): break
    except Exception: pass
    time.sleep(1)
else:
    raise RuntimeError("Comfy no estuvo listo: "+logfile.read_text(errors="replace")[-6000:])
print("Comfy listo. PID:",p.pid)
print("Health:",json.dumps(health,ensure_ascii=False,indent=2))
print("Log:",logfile)

In [ ]:
# 4 — Abrir interfaz e inspeccionar 6 nodos Context
from google.colab import output
output.serve_kernel_port_as_iframe(8188, height=880)
print("Al abrir Comfy: Auto Queue OFF; Settings → Workflow Tabs Position → Topbar.")
print("Comprueba que existe Workflow Director Lab en el panel inferior.")

In [ ]:
# 5 — Preflight REAL de API, nodos, Comfy y --cache-none
PROBE = WD_DIR / "scripts" / "acceptance_probe.py"
subprocess.run([sys.executable,str(PROBE),"--case","preflight"],check=True)

## Pruebas automáticas livianas

**Ejecuta cada celda en orden y detente si una falla.** Todas son jobs nativos reales de Comfy, pero no descargan ni cargan modelos GGUF. Se guardará un reporte acumulativo en `/content/workflowdirector_acceptance_report.json`.

**STRING** publica un texto desde A y B lo consume en otro job.

**IMAGE** crea una imagen simple de 96×64 y B la guarda con prefijo `WD_Acceptance_IMAGE_`.

**LATENT** crea un latent 64×64 y B lo guarda con prefijo `WD_Acceptance_LATENT_`.

**FAILURE** fuerza un error en A. El criterio de éxito es que el Director marque el run como failed y **no** envíe B.

In [ ]:
# 6 — STRING: A produce un dato; B lo lee y envía al Marker
subprocess.run([sys.executable,str(PROBE),"--case","string"],check=True)

In [ ]:
# 7 — IMAGE: produce 96×64; B guarda una imagen real
subprocess.run([sys.executable,str(PROBE),"--case","image"],check=True)

In [ ]:
# 8 — LATENT: produce latent sin modelo; B lo guarda
subprocess.run([sys.executable,str(PROBE),"--case","latent"],check=True)

In [ ]:
# 9 — FALLA INTENCIONAL: A falla y B NO debe iniciar
subprocess.run([sys.executable,str(PROBE),"--case","failure"],check=True)

## Verificación visual del panel

Después de pasar las cuatro pruebas anteriores:

1. Descarga los dos JSON del ejemplo STRING con la celda siguiente.
2. Abre **cada JSON como pestaña distinta**, con Tabs Position = Topbar, Auto Queue apagado.
3. En la pestaña A pulsa **Capture current as A**, después en B **Capture current as B**.
4. Deja marcadas **Refresh linked tabs before Run**, **Show executing workflow tab** y **Notify on each workflow**.
5. Modifica en A el texto del nodo PutString después de capturarlo, **sin volver a pulsar Capture A**.
6. Pulsa **Run A → B**. Debe salir `completed`, mostrar `greeting` en el Context, ejecutar B con el texto nuevo y cambiar de pestaña con avisos. Comprueba el texto recibido en el log del nodo Marker.
7. Después cierra B e intenta **Run A → B**. Debe rechazar la operación antes de enviar cualquier job; si no, detén las pruebas.

Las capturas están en la memoria del navegador; si recargas Comfy, vuelve a capturar A/B.

In [ ]:
# 10 — Descargar workflows de ejemplo A y B
from google.colab import files
for name in ("context_string_A.json","context_string_B.json"):
    path=WD_DIR/"workflows"/name
    assert path.exists()
    files.download(str(path))

In [ ]:
# 11 — Diagnóstico y evidencia (se puede volver a ejecutar)
import json
from urllib.request import urlopen
for path in ("/workflowdirector/health","/workflowdirector/memory",
             "/api/jobs?status=pending,in_progress"):
    with urlopen(BASE+path,timeout=10) as r:
        value=json.load(r)
    print("\n",path)
    if path.endswith("/memory"):
        print("RSS:",value.get("process_rss_gib"),"PSS:",value.get("process_memory"))
    else:
        print(json.dumps(value,ensure_ascii=False,indent=2)[:3500])
report=Path("/content/workflowdirector_acceptance_report.json")
if report.exists():
    print("\nACCEPTANCE REPORT:",report.read_text()[-13000:])
log=Path("/content/comfyui_workflowdirector_acceptance.log")
if log.exists():
    print("\nCOMFY LOG TAIL:",log.read_text(errors="replace")[-7500:])

In [ ]:
# 12 — Descargar evidencia para revisar y compartir
import zipfile, datetime
report=Path("/content/workflowdirector_acceptance_report.json")
log=Path("/content/comfyui_workflowdirector_acceptance.log")
stamp=datetime.datetime.now().strftime("%Y%m%d_%H%M")
bundle=Path(f"/content/workflowdirector_acceptance_{stamp}.zip")
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    if report.exists(): z.write(report,arcname="acceptance_report.json")
    if log.exists():
        excerpt=log.read_text(errors="replace")[-90000:]
        # Logs may include local paths or prompts. Inspect before sharing.
        z.writestr("comfy_log_tail.txt",excerpt)
    z.writestr("workflowdirector_commit.txt",run_git("rev-parse","HEAD"))
    z.writestr("original_commit.txt",Path("/content/workflowdirector_before_acceptance_commit.txt").read_text())
print(bundle)
print("Revisa el ZIP antes de compartirlo; puede contener texto de prompts y rutas.")
files.download(str(bundle))

## Restauración (opcional, sin borrar nada)

Si deseas regresar al WorkflowDirector anterior, ejecuta la siguiente celda **cuando no haya jobs**. Después vuelve a ejecutar la celda 3 para reiniciar Comfy con el código restaurado. Esta restauración NO desinstala GGUF ni cambia versiones de Comfy/Torch.

In [ ]:
# 13 — Restaurar WorkflowDirector anterior SOLO cuando lo decidas
# Mantener comentado por defecto; quitar el comentario de la última línea para ejecutar.
def restore_workflowdirector():
    with urlopen(BASE+"/api/jobs?status=pending,in_progress",timeout=10) as r:
        assert not json.load(r).get("jobs"), "No restaurar durante una ejecución."
    original=Path("/content/workflowdirector_before_acceptance_commit.txt").read_text().strip()
    assert not run_git("status","--porcelain"), "Hay cambios locales: conservarlos."
    subprocess.run(["git","-C",str(WD_DIR),"checkout","--detach",original],check=True)
    print("Código restaurado:",run_git("rev-parse","--short","HEAD"))
    print("Reinicia Comfy con tu mismo comando de arranque.")
# restore_workflowdirector()